# 29. Anotación, Fase 3: la referencia final de OE1

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`04-jss-anotacion-f1-calibracion.ipynb`](04-jss-anotacion-f1-calibracion.ipynb) ·
[`18-jss-anotacion-f2-adjudicacion.ipynb`](18-jss-anotacion-f2-adjudicacion.ipynb) ·
[`26-jss-minimo-e3-relaciones-disponibles.ipynb`](26-jss-minimo-e3-relaciones-disponibles.ipynb) ·
[`27-jss-relaciones-con-extremo-compuesto.ipynb`](27-jss-relaciones-con-extremo-compuesto.ipynb) ·
[`src/afg/evaluation/reference.py`](../src/afg/evaluation/reference.py) ·
[`src/afg/extraction/openkos/gold_bundle.py`](../src/afg/extraction/openkos/gold_bundle.py) ·
[`src/afg/annotation/recall.py`](../src/afg/annotation/recall.py)

**Estado: completo. Cierra la anotación humana de OE1.** Describe la referencia tal como queda
después de la Fase 3: las 14 series (3 de desarrollo y 11 de evaluación), cuántas decisiones y
cuántas relaciones tiene cada una, quién produjo cada referencia y el resultado de la muestra de
recall de la Tarea C de TS3005 (issue #194).

---

> **Aviso de alcance (ADR 0005).** El notebook muestra solo **identificadores de serie y
> conteos**. No muestra ni cita frases de decisión, evidencias, notas ni razones de
> adjudicación, y de las muestras de recall solo lee la columna de etiqueta. No contiene ninguna
> medida del Experimento A ni del B.

# 1. Carga

Las cifras salen de los mismos archivos que usa el paquete:

- **Decisiones:** el archivo que `resolve_reference` elige como referencia de cada serie
  (`<SERIE>.decisions.adjudicated.csv` o, sin adjudicación, el único `<SERIE>.decisions.<ini>.csv`).
  La columna de estado final es `final_status` en el archivo adjudicado y `status` en el
  individual.
- **Relaciones:** `load_gold_relations`, que devuelve las relaciones aceptadas de la Tarea B
  (etiqueta distinta de `no_relacionada` con la dirección confirmada).
- **Base de la máquina:** `<SERIE>.decisions.csv`, la lista de frases candidatas que las personas
  etiquetaron; sirve para reconciliar con las cifras de los documentos 02 y 08.

In [1]:
import re

import pandas as pd

from afg.annotation.recall import recall_report
from afg.evaluation.reference import resolve_reference
from afg.experiments.question_bank import evaluation_series
from afg.extraction.openkos.gold_bundle import gold_relations_kind, load_gold_relations
from afg.shared.config import load_corpus_config
from afg.shared.paths import GOLD_DECISIONS_DIR, GOLD_RELATIONS_DIR, PROJECT_ROOT

OE1 = load_corpus_config()["oe1"]
ALL_SERIES = tuple(OE1["series"])
DEVELOPMENT = tuple(OE1["development_series"])
EVALUATION = evaluation_series()
STATUS_COLUMNS = ("final_status", "status")
EVOLUTION = ("refina", "revierte", "reemplaza")
TYPES = ("reafirma", "refina", "revierte", "reemplaza")

assert set(EVALUATION) | set(DEVELOPMENT) == set(ALL_SERIES)
print(
    f"Series: {len(ALL_SERIES)} = {len(DEVELOPMENT)} de desarrollo "
    f"+ {len(EVALUATION)} de evaluación"
)
print("Desarrollo:", ", ".join(DEVELOPMENT))
print("Evaluación:", ", ".join(EVALUATION))

Series: 14 = 3 de desarrollo + 11 de evaluación
Desarrollo: ES2015, IS1004, TS3009
Evaluación: ES2002, ES2008, ES2014, ES2016, IS1003, IS1006, IS1008, IS1009, TS3003, TS3005, TS3011


# 2. Decisiones por serie

Cada fila de la referencia tiene un estado final. Solo las de estado `decision` forman la
referencia; las de estado `no_decision`, `descartar` y `sin_soporte` se excluyen.

**Cómo se cuentan las decisiones compuestas.** Una frase marcada `compuesta` agrupa varias
decisiones. Se reemplaza por sus hijas (`d07-1`, `d07-2`, ...), cada una con estado `decision`.
`load_reference` deja fuera al **padre** y conserva a las **hijas**, de modo que la columna
«decisiones» cuenta hijas y no padres. La relación entre la base y la referencia es entonces:

`decisiones = base − padres compuestos − excluidas por estado + hijas`

In [2]:
def read_status_frame(series):
    ref = resolve_reference(series)
    frame = pd.read_csv(ref.path, dtype=str, keep_default_na=False)
    column = next(c for c in STATUS_COLUMNS if c in frame.columns)
    return ref.kind, frame["decision_id"].tolist(), frame[column].str.strip().tolist()


def base_rows(series):
    return len(pd.read_csv(GOLD_DECISIONS_DIR / f"{series}.decisions.csv", dtype=str))


rows = []
PARENTS = {}
for s in ALL_SERIES:
    kind, ids, statuses = read_status_frame(s)
    parents = [i for i, st in zip(ids, statuses, strict=True) if st == "compuesta"]
    PARENTS[s] = set(parents)
    children = [i for i in ids if any(re.fullmatch(re.escape(p) + r"-\d+", i) for p in parents)]
    excluded = {st: statuses.count(st) for st in ("no_decision", "descartar", "sin_soporte")}
    decisions = statuses.count("decision")
    base = base_rows(s)
    assert decisions == base - len(parents) - sum(excluded.values()) + len(children), s
    rows.append(
        {
            "serie": s,
            "grupo": "desarrollo" if s in DEVELOPMENT else "evaluación",
            "origen": "adjudicada"
            if kind == "adjudicated"
            else f"individual ({kind.split(':')[1]})",
            "base": base,
            "padres compuestos": len(parents),
            "hijas": len(children),
            "no_decision": excluded["no_decision"],
            "descartar": excluded["descartar"],
            "sin_soporte": excluded["sin_soporte"],
            "decisiones": decisions,
        }
    )
DECISIONS = pd.DataFrame(rows).set_index("serie")
DECISIONS

,grupo,origen,base,padres compuestos,hijas,no_decision,descartar,sin_soporte,decisiones
serie,,,,,,,,,
ES2002,evaluación,individual (gv),28,1,4,0,0,0,31
ES2008,evaluación,adjudicada,33,1,4,4,0,0,32
ES2014,evaluación,individual (gm),23,5,17,1,1,1,32
ES2015,desarrollo,adjudicada,29,2,5,3,1,1,27
ES2016,evaluación,adjudicada,24,1,2,3,0,1,21
IS1003,evaluación,adjudicada,25,0,0,2,1,0,22
IS1004,desarrollo,individual (gv),24,2,8,1,1,1,27
IS1006,evaluación,individual (gv),17,0,0,1,0,0,16
IS1008,evaluación,individual (gm),13,3,14,10,1,2,11


In [3]:
numeric = [
    "base",
    "padres compuestos",
    "hijas",
    "no_decision",
    "descartar",
    "sin_soporte",
    "decisiones",
]
by_group = DECISIONS.groupby("grupo")[numeric].sum()
by_group.loc["total (14 series)"] = DECISIONS[numeric].sum()
by_group

,base,padres compuestos,hijas,no_decision,descartar,sin_soporte,decisiones
grupo,,,,,,,
desarrollo,77,6,18,5,2,2,80
evaluación,266,18,70,29,6,5,278
total (14 series),343,24,88,34,8,7,358


**Reconciliación con las cifras de los documentos.**

- **343** (documentos 02 y 03, `config/corpus.toml`) es el tamaño de la **base**: las frases
  `DECISIONS` de los resúmenes del corpus en las 14 series, antes de que nadie las etiquete. La
  columna «base» suma 343.
- **278** (documento 11) son las decisiones de la **referencia** de las 11 series de evaluación.
- La referencia completa de las 14 series tiene **358** decisiones. La diferencia con 343 no es
  un cambio de muestra: es el resultado del etiquetado. Las personas descartaron frases que no
  eran decisiones (columnas `no_decision`, `descartar`, `sin_soporte`), y reemplazaron cada
  padre compuesto por sus hijas, que son más numerosas que los padres.
- 358 menos las 80 decisiones de las tres series de desarrollo (ES2015, IS1004, TS3009) da las
  278 de evaluación.

In [4]:
n_base = int(DECISIONS["base"].sum())
n_ref = int(DECISIONS["decisiones"].sum())
n_ref_eval = int(DECISIONS.loc[list(EVALUATION), "decisiones"].sum())
n_ref_dev = int(DECISIONS.loc[list(DEVELOPMENT), "decisiones"].sum())
print(f"Base (14 series): {n_base}")
print(f"Referencia (14 series): {n_ref} = {n_ref_dev} de desarrollo + {n_ref_eval} de evaluación")
assert n_base == 343 and n_ref_eval + n_ref_dev == n_ref

Base (14 series): 343
Referencia (14 series): 358 = 80 de desarrollo + 278 de evaluación


# 3. Relaciones de oro por serie y tipo

Una relación de oro es la que `load_gold_relations` acepta. Se cuentan por tipo: `reafirma` es
una relación de continuidad y `refina`, `revierte` y `reemplaza` son las relaciones de evolución.
Además se cuentan las relaciones cuyo extremo es un padre `compuesto` (columna «extremo
compuesto»): la corrección del [notebook 27](27-jss-relaciones-con-extremo-compuesto.ipynb) (ADR
0025) alcanzó solo a las 11 series de evaluación, así que la celda exige 0 allí y solo informa el
valor en las de desarrollo.

In [5]:
rows = []
for s in ALL_SERIES:
    relations = load_gold_relations(s)
    parents = PARENTS[s]
    composite_end = sum(r.earlier_id in parents or r.later_id in parents for r in relations)
    assert s in DEVELOPMENT or composite_end == 0, s
    counts = {t: sum(r.relation == t for r in relations) for t in TYPES}
    assert sum(counts.values()) == len(relations), s
    rows.append(
        {
            "serie": s,
            "grupo": "desarrollo" if s in DEVELOPMENT else "evaluación",
            "origen": gold_relations_kind(s),
            **counts,
            "relaciones": len(relations),
            "de evolución": sum(counts[t] for t in EVOLUTION),
            "extremo compuesto": composite_end,
        }
    )
RELATIONS = pd.DataFrame(rows).set_index("serie")
RELATIONS

,grupo,origen,reafirma,refina,revierte,reemplaza,relaciones,de evolución,extremo compuesto
serie,,,,,,,,,
ES2002,evaluación,single:gv,1,1,1,0,3,2,0
ES2008,evaluación,adjudicated,0,5,0,0,5,5,0
ES2014,evaluación,single:gm,3,3,0,0,6,3,0
ES2015,desarrollo,adjudicated,2,1,0,0,3,1,0
ES2016,evaluación,adjudicated,2,2,0,0,4,2,0
IS1003,evaluación,adjudicated,0,2,0,1,3,3,0
IS1004,desarrollo,single:gv,0,3,0,0,3,3,2
IS1006,evaluación,single:gv,0,0,0,0,0,0,0
IS1008,evaluación,single:gm,0,1,0,0,1,1,0


In [6]:
cols = [*TYPES, "relaciones", "de evolución", "extremo compuesto"]
REL_TOTALS = RELATIONS.groupby("grupo")[cols].sum()
REL_TOTALS.loc["total (14 series)"] = RELATIONS[cols].sum()
display(REL_TOTALS)

rel_eval = int(RELATIONS.loc[list(EVALUATION), "relaciones"].sum())
evo_eval = int(RELATIONS.loc[list(EVALUATION), "de evolución"].sum())
print(f"Relaciones de oro en las 11 series de evaluación: {rel_eval}")
print(f"De evolución (refina, revierte, reemplaza) en las 11 series de evaluación: {evo_eval}")
assert rel_eval == 37 and evo_eval == 24

,reafirma,refina,revierte,reemplaza,relaciones,de evolución,extremo compuesto
grupo,,,,,,,
desarrollo,2,6,0,1,9,7,3
evaluación,13,20,3,1,37,24,0
total (14 series),15,26,3,2,46,31,3


Relaciones de oro en las 11 series de evaluación: 37
De evolución (refina, revierte, reemplaza) en las 11 series de evaluación: 24


**Reconciliación con los notebooks 26 y 27.**

- **37** es la cifra del notebook 27: relaciones aceptadas de la Tarea B en las 11 series de
  evaluación. Aquí es la suma de la fila «evaluación».
- **24** es la cifra del notebook 26: las relaciones de evolución (sin contar `reafirma`) de esas
  mismas 11 series. Aquí es la columna «de evolución» de la fila «evaluación». Las 13 restantes
  son `reafirma`.
- Las tres series de desarrollo aportan 9 relaciones más, hasta 46 en las 14 series. Esas 9 no
  entran en el Experimento A ni en el mínimo de E3, porque las series de desarrollo no se
  evalúan (ADR 0005).
- **Extremos compuestos en desarrollo.** 3 de esas 9 relaciones (2 en IS1004 y 1 en TS3009)
  siguen apuntando a un padre `compuesto`, porque la reasignación del ADR 0025 se limitó a las
  series de evaluación. No afectan a ninguna cifra de evaluación; se declaran aquí para que la
  composición de las 14 series sea exacta.

# 4. Quién produjo cada referencia

La referencia de una serie es una de dos cosas:

- **Adjudicada:** `gm` y `gv` anotaron por separado, y `jss` resolvió cada desacuerdo y dejó la
  razón escrita. Es el diseño de las Fases 1 y 2.
- **Individual (`single:gv` o `single:gm`):** una sola persona anotó la serie (Fase 3) y su
  archivo es la referencia. Una serie individual no tiene kappa, porque no hay segunda lectura.

Las decisiones y las relaciones de cada serie tienen el mismo origen.

In [7]:
origin = DECISIONS[["grupo", "origen"]].copy()
origin["origen de relaciones"] = RELATIONS["origen"].map(
    lambda k: "adjudicada" if k == "adjudicated" else f"individual ({k.split(':')[1]})"
)
assert (origin["origen"] == origin["origen de relaciones"]).all()

split = (
    DECISIONS.assign(relaciones=RELATIONS["relaciones"])
    .groupby(["grupo", "origen"])[["decisiones", "relaciones"]]
    .agg(["count", "sum"])
)
summary = pd.DataFrame(
    {
        "series": split[("decisiones", "count")],
        "decisiones": split[("decisiones", "sum")],
        "relaciones": split[("relaciones", "sum")],
    }
)
display(summary)

lists = origin.reset_index().groupby(["grupo", "origen"])["serie"].apply(lambda s: ", ".join(s))
display(lists.to_frame("series"))

series  decisiones  relaciones
grupo      origen                                         
desarrollo adjudicada            2          53           6
           individual (gv)       1          27           3
evaluación adjudicada            4         112          20
           individual (gm)       3          64           8
           individual (gv)       4         102           9

series
grupo      origen                                         
desarrollo adjudicada                       ES2015, TS3009
           individual (gv)                          IS1004
evaluación adjudicada       ES2008, ES2016, IS1003, TS3005
           individual (gm)          ES2014, IS1008, TS3011
           individual (gv)  ES2002, IS1006, IS1009, TS3003

In [8]:
n_adj = int((DECISIONS.loc[list(EVALUATION), "origen"] == "adjudicada").sum())
print(
    f"Series de evaluación con doble anotación y adjudicación: {n_adj} de {len(EVALUATION)} "
    f"({n_adj / len(EVALUATION):.1%})"
)

Series de evaluación con doble anotación y adjudicación: 4 de 11 (36.4%)


Las cuatro series de evaluación adjudicadas son las de control de la Fase 2 (ES2016, IS1003,
TS3005 y ES2008), el 36,4 % de la evaluación, que cumple el criterio de doble anotación de al
menos el 25 % ([documento 10](../docs/avances/10-jss-confiabilidad-de-la-referencia.md)). Las
otras siete series de evaluación tienen un único anotador: `gv` en ES2002, IS1006, IS1009 y
TS3003, y `gm` en ES2014, IS1008 y TS3011. De desarrollo, ES2015 y TS3009 son adjudicadas
(Fase 1) e IS1004 es individual (`gv`).

# 5. Muestra de recall de la Tarea C de TS3005 (issue #194)

El bloqueador léxico descarta los pares cuyo solapamiento queda bajo el umbral. La Tarea C
etiqueta una muestra aleatoria de 50 pares rechazados por serie (semilla 42); cualquier etiqueta
distinta de `no_relacionada` es una relación real que el bloqueador perdió (un falso negativo).
El issue #194 pedía la muestra de TS3005, la última pendiente de la Fase 2 para el sitio TS, y
quedó cerrado con la entrega de `TS3005.recall-sample.gv.csv`.

El estimador y su intervalo (Wilson al 95 %) son los de `recall_report`, los mismos del
[notebook 18, sección 8](18-jss-anotacion-f2-adjudicacion.ipynb). Aquí se muestra solo la fila
de TS3005 y la del total, que cierra con las cuatro muestras (IS1004, ES2016, IS1003 y TS3005).

In [9]:
RECALL_SERIES = ["IS1004", "ES2016", "IS1003", "TS3005"]
RECALL = recall_report(RECALL_SERIES, "gv")
pending = ", ".join(RECALL.missing) or "ninguna"
print(f"Muestras de recall usadas: {RECALL.n_used} de {RECALL.n_planned}; pendientes: {pending}")


def recall_row(e):
    return {
        "grupo": e.label,
        "n muestra": e.n_sampled,
        "falsos negativos": e.n_false_negative,
        "pares rechazados": e.rejected_population,
        "relaciones conservadas": e.kept_relations,
        "tasa de FN": round(e.false_negative_rate, 3),
        "IC 95 % tasa": f"[{e.rate_lower:.3f}; {e.rate_upper:.3f}]",
        "perdidas estimadas": round(e.missed, 1),
        "recall": round(e.recall, 3),
        "IC 95 % recall": f"[{e.recall_lower:.3f}; {e.recall_upper:.3f}]",
    }


ts = next(e for e in RECALL.series if e.label == "TS3005")
display(pd.DataFrame([recall_row(ts), recall_row(RECALL.total)]).set_index("grupo"))

Muestras de recall usadas: 4 de 4; pendientes: ninguna


,n muestra,falsos negativos,pares rechazados,relaciones conservadas,tasa de FN,IC 95 % tasa,perdidas estimadas,recall,IC 95 % recall
grupo,,,,,,,,,
TS3005,50,3,254,8,0.060,[0.021; 0.162],15.2,0.344,[0.163; 0.604]
Total,200,6,838,18,0.031,[0.015; 0.066],26.3,0.407,[0.245; 0.596]


In [10]:
# Solo se lee la columna de etiqueta de la muestra de TS3005 (ADR 0005): ninguna columna de texto.
SAMPLE_PATH = GOLD_RELATIONS_DIR / "TS3005.recall-sample.gv.csv"
sample = pd.read_csv(SAMPLE_PATH, usecols=["relation"], dtype=str, keep_default_na=False)
labels = sample["relation"].str.strip().replace("", "(sin etiqueta)")
display(labels.value_counts().rename_axis("etiqueta").to_frame("pares"))
assert len(sample) == ts.n_sampled == 50 and (labels != "(sin etiqueta)").all()

,pares
etiqueta,
no_relacionada,47
refina,2
reemplaza,1


**Lectura.**

- La muestra de TS3005 está completa: 50 de 50 pares etiquetados. 3 son relaciones reales que el
  bloqueador descartó (2 `refina` y 1 `reemplaza`) y los otros 47 son `no_relacionada`. La tasa de
  falsos negativos de la muestra es 0,060 [0,021; 0,162].
- TS3005 tiene 254 pares rechazados y 8 relaciones conservadas en la Tarea B adjudicada. Con 3 de
  50 en la muestra se estiman unas 15,2 relaciones perdidas, y el recall de la serie queda en
  0,344 [0,163; 0,604]. Es la misma cifra que da el notebook 18.
- Con las cuatro muestras (200 pares, 6 falsos negativos) el recall total es 0,407
  [0,245; 0,596], también el del notebook 18. Este notebook no agrega muestras: registra que la de
  TS3005 está entregada y completa, de modo que ya no hay muestras pendientes.
- El intervalo es ancho (50 pares por serie): la cifra es una estimación, no una medida precisa.
  Qué hacer con esa pérdida lo decide el [notebook 25](25-jss-canal-semantico-del-bloqueador.ipynb)
  y su ADR, no este notebook.

# 6. Confiabilidad: los kappas ya son finales

Los kappas de OE1 se calculan solo donde hay dos lecturas independientes: en la Fase 1
([notebook 04](04-jss-anotacion-f1-calibracion.ipynb), diagnóstico, no se reporta) y en la Fase 2
([notebook 18](18-jss-anotacion-f2-adjudicacion.ipynb), reportado). La Fase 3 es de anotación
individual y **no agrega ningún kappa**. La tabla siguiente se lee de
`reports/tables/agreement-pooled.csv`, el archivo que generan esas dos fases, y no se recalcula.
Se muestra la Fase 2, la única que se reporta: existencia 0,783 sobre 37 pares, tipo 0,554 sobre
18 y dirección indefinida (sin variación). La fila `status` es el kappa de estado, que es
diagnóstico y no se reporta como confiabilidad (manual de anotación, §6).

In [11]:
POOLED = pd.read_csv(PROJECT_ROOT / "reports" / "tables" / "agreement-pooled.csv")
phase2 = POOLED[POOLED["scope"] == "phase 2"]
shown = phase2[["axis", "n_items", "n_disagreements", "kappa", "percent_agreement"]]
display(shown.set_index("axis"))
assert set(POOLED["scope"]) == {"overall", "phase 1", "phase 2"}

,n_items,n_disagreements,kappa,percent_agreement
axis,,,,
status,111,21,-0.054274,0.810811
existence,37,4,0.782991,0.891892
type,18,5,0.554455,0.722222
direction,37,0,NaN,1.000000


# 7. Lectura y límites

- **Composición final.** La referencia de OE1 son 14 series con 358 decisiones y 46 relaciones de
  oro. De ellas, 11 series (278 decisiones y 37 relaciones, 24 de evolución) son de evaluación y
  3 (80 decisiones y 9 relaciones) son de desarrollo.
- **Origen mixto.** Seis de las 14 series son adjudicadas tras doble anotación (cuatro de
  evaluación) y ocho tienen un solo anotador. Las cifras de confiabilidad de OE1 describen las
  series de control de la Fase 2; la parte de la referencia con un solo anotador no tiene medida
  de acuerdo, y esa es una limitación que la tesis debe declarar.
- **Las 343 y las 358.** Son magnitudes distintas (base de la máquina y referencia humana) y no
  se contradicen.
- **Sin contenido.** Como las series de evaluación son de evaluación, el notebook no muestra
  frases ni casos (ADR 0005).